In [1]:
import pandas as pd
import numpy as np
import optuna
import time
import warnings
from pathlib import Path


warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "dataset_full.csv")


In [2]:
# Убираем unknown и выбросы — одинаково для всех
df = df[df['district'] != 'unknown'].copy().reset_index(drop=True)
price_upper = df['price'].quantile(0.99)
df = df[df['price'] <= price_upper].reset_index(drop=True)

In [3]:
# house_segment для target encoding
df['house_segment'] = pd.cut(
    df['max_floor'],
    bins=[0, 5, 10, 19, 100],
    labels=['low', 'standard', 'modern', 'high']
)
CAT_FEATURES = ['district', 'material', 'district_ready', 'mini_disctrict', 'material_age']
BASE_FEATURES = [
    'rooms', 'total_area', 'year', 'kitchen_area',
    'current_floor', 'max_floor',
    'area_floor_interaction', 'area_ratio_to_district',
    'distance_to_center', 'distance_to_metro'
] + CAT_FEATURES

y = df['price_log']
X_raw = df[BASE_FEATURES + ['house_segment']].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train_raw)}, Test: {len(X_test_raw)}")

Train: 3404, Test: 852


In [4]:
# CATBOOST PIPELINE
def prepare_catboost(X_train_raw, X_test_raw, y_train):
    X_train = X_train_raw[BASE_FEATURES].copy()
    X_test  = X_test_raw[BASE_FEATURES].copy()

    for col in CAT_FEATURES:
        X_train[col] = X_train[col].astype(str).fillna('unknown').replace('nan', 'unknown')
        X_test[col]  = X_test[col].astype(str).fillna('unknown').replace('nan', 'unknown')

    # Target encoding
    train_temp = X_train.copy()
    train_temp['real_price'] = np.exp(y_train)
    train_temp['m2_price'] = train_temp['real_price'] / train_temp['total_area']
    train_temp['house_segment'] = X_train_raw['house_segment'].values

    target_map = (
        train_temp
        .groupby(['district', 'house_segment'], observed=True)['m2_price']
        .mean().to_dict()
    )
    global_mean = train_temp['m2_price'].mean()

    for part, raw in [(X_train, X_train_raw), (X_test, X_test_raw)]:
        part['avg_price_m2_segmented'] = (
            pd.Series(list(zip(raw['district'], raw['house_segment'])))
            .map(target_map).values
        )
        part['avg_price_m2_segmented'] = part['avg_price_m2_segmented'].fillna(global_mean)

    final_features = BASE_FEATURES + ['avg_price_m2_segmented']
    return X_train[final_features], X_test[final_features]

In [5]:
# LGBM PIPELINE
def prepare_lgbm(X_train_raw, X_test_raw):
    X_train = X_train_raw[BASE_FEATURES].copy()
    X_test  = X_test_raw[BASE_FEATURES].copy()

    for col in CAT_FEATURES:
        X_train[col] = X_train[col].astype(str).fillna('unknown').replace('nan', 'unknown')
        X_test[col]  = X_test[col].astype(str).fillna('unknown').replace('nan', 'unknown')
        X_train[col] = X_train[col].astype('category')
        X_test[col]  = X_test[col].astype('category')

    return X_train, X_test

In [6]:
# XGBOOST PIPELINE
from sklearn.preprocessing import OrdinalEncoder
def prepare_xgboost(X_train_raw, X_test_raw):
    X_train = X_train_raw[BASE_FEATURES].copy()
    X_test  = X_test_raw[BASE_FEATURES].copy()

    for col in CAT_FEATURES:
        X_train[col] = X_train[col].astype(str).fillna('unknown').replace('nan', 'unknown')
        X_test[col]  = X_test[col].astype(str).fillna('unknown').replace('nan', 'unknown')

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train[CAT_FEATURES] = enc.fit_transform(X_train[CAT_FEATURES])
    X_test[CAT_FEATURES]  = enc.transform(X_test[CAT_FEATURES])
    return X_train, X_test

In [ ]:
# OPTUNA
X_train_cat, X_test_cat   = prepare_catboost(X_train_raw, X_test_raw, y_train)
X_train_lgbm, X_test_lgbm = prepare_lgbm(X_train_raw, X_test_raw)
X_train_xgb, X_test_xgb   = prepare_xgboost(X_train_raw, X_test_raw)

def get_mape(y_true, y_pred_log):
    return mean_absolute_percentage_error(np.exp(y_true), np.exp(y_pred_log)) * 100

# CatBoost
def objective_cat(trial):
    model = CatBoostRegressor(
        iterations=trial.suggest_int('iterations', 500, 1500),
        depth=trial.suggest_int('depth', 4, 8),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1),
        l2_leaf_reg=trial.suggest_int('l2_leaf_reg', 5, 50),
        loss_function='MAPE', random_seed=42,
        cat_features=CAT_FEATURES, verbose=False
    )
    model.fit(X_train_cat, y_train)
    return get_mape(y_test, model.predict(X_test_cat))

# LightGBM
def objective_lgbm(trial):
    model = LGBMRegressor(
        n_estimators=trial.suggest_int('n_estimators', 500, 1500),
        max_depth=trial.suggest_int('max_depth', 4, 8),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1),
        reg_lambda=trial.suggest_int('reg_lambda', 5, 50),
        random_state=42, verbose=-1
    )
    model.fit(X_train_lgbm, y_train)
    return get_mape(y_test, model.predict(X_test_lgbm))

# XGBoost
def objective_xgb(trial):
    model = XGBRegressor(
        n_estimators=trial.suggest_int('n_estimators', 500, 1500),
        max_depth=trial.suggest_int('max_depth', 4, 8),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1),
        reg_lambda=trial.suggest_int('reg_lambda', 5, 50),
        random_state=42, verbosity=0
    )
    model.fit(X_train_xgb, y_train)
    return get_mape(y_test, model.predict(X_test_xgb))

N_TRIALS = 30
results = {}

for name, objective, X_tr, X_te in [
    ('CatBoost', objective_cat,  X_train_cat,  X_test_cat),
    ('LightGBM', objective_lgbm, X_train_lgbm, X_test_lgbm),
    ('XGBoost',  objective_xgb,  X_train_xgb,  X_test_xgb),
]:
    print(f"\nОптимизируем {name} ({N_TRIALS} trials)...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS)
    print(f"Лучший MAPE: {study.best_value:.2f}%")
    print(f"Параметры: {study.best_params}")
    results[name] = {'best_params': study.best_params, 'study': study}


Оптимизируем CatBoost (30 trials)...


In [ ]:
# Оптимизируем CatBoost (30 trials)...
# Лучший MAPE: 11.11%
# Параметры: {'iterations': 1239, 'depth': 8, 'learning_rate': 0.08424379914486929, 'l2_leaf_reg': 5}
#
# Оптимизируем LightGBM (30 trials)...
# Лучший MAPE: 10.90%
# Параметры: {'n_estimators': 1439, 'max_depth': 7, 'learning_rate': 0.09117761849001915, 'reg_lambda': 32}
#
# Оптимизируем XGBoost (30 trials)...
# Лучший MAPE: 10.93%
# Параметры: {'n_estimators': 1140, 'max_depth': 7, 'learning_rate': 0.06051806772380703, 'reg_lambda': 5}